# Week 1.3: Attention — how a model works out what you mean
### Plain-English edition — no coding required

Read this sentence: **"I sat on the bank of the river."**

You knew instantly that *bank* means a muddy edge, not a place that keeps money.
Nobody told you. You worked it out from the word **river** sitting nearby.

That trick — letting every word look at the other words around it — is called
**attention**, and it is the single idea that made ChatGPT possible.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## What you will learn
- Why the same word can mean two different things
- What a model actually *looks at* when reading your sentence
- How to read an attention map
- Why several "heads" read the same sentence in different ways

---
## Part 1: The problem — one word, two meanings

Before attention, a model gave each word one fixed meaning. So *bank* was always
the same *bank*, no matter what surrounded it. Watch that go wrong.

In [ ]:
#@title ▶ Step 1 — Show me the problem with one word, two meanings { display-mode: "form" }
#@markdown Pick a word that has more than one meaning, and write two sentences using it.
tricky_word = "bank" #@param {type:"string"}
sentence_one = "I went to the bank to deposit money" #@param {type:"string"}
sentence_two = "I sat on the bank of the river" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

# a small, meaningful word space so attention patterns are readable
CORPUS = [
 "went to the bank to deposit money in my account",
 "the bank gave me a loan and i paid money into the account",
 "money cash account loan deposit bank branch",
 "sat on the bank of the river watching the water",
 "the river bank was steep grassy and muddy",
 "river water fish boat grassy muddy bank",
 "the dog chased the cat around the garden",
 "my dog barked at the cat in the garden",
 "dog cat pet animal garden chased barked",
 "i love running football swimming and sport",
 "running sport exercise football healthy",
 "she ate rice and chicken for her lunch",
 "rice chicken food lunch tasty eating",
]
STOP = {"the","a","an","to","of","in","on","at","and","my","i","it","was","is","for","her","me"}
def content(text):
    keep = [w for w in words_of(text) if w not in STOP]
    return keep if len(keep) >= 4 else words_of(text)
def build_space(dim=16, window=4):
    T=[content(c) for c in CORPUS]
    vocab=sorted({w for t in T for w in t}); idx={w:i for i,w in enumerate(vocab)}
    nv=len(vocab); M=np.zeros((nv,nv))
    for t in T:
        for i,w in enumerate(t):
            for j in range(max(0,i-window), min(len(t),i+window+1)):
                if i!=j: M[idx[w],idx[t[j]]]+=1.0
    tot=M.sum(); rows=M.sum(1,keepdims=True); cols=M.sum(0,keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P=np.log((M*tot)/(rows*cols+1e-12)+1e-12)
    P[~np.isfinite(P)]=0; P[P<0]=0
    U,S,Vt=np.linalg.svd(P, full_matrices=False)
    V=U[:,:dim]*S[:dim]
    return vocab, idx, V/(np.linalg.norm(V,axis=1,keepdims=True)+1e-9)
def space():
    if "_SP3" not in globals(): globals()["_SP3"]=build_space()
    return globals()["_SP3"]
def attention(tokens):
    # how much each word looks at every other word: meaning similarity + nearness in the sentence
    vocab, idx, V = space()
    nT=len(tokens)
    vecs=np.array([V[idx[t]] if t in idx else np.zeros(V.shape[1]) for t in tokens])
    sim=vecs.dot(vecs.T)
    pos=np.array([[1.0/(1.0+abs(i-j)) for j in range(nT)] for i in range(nT)])
    raw=2.2*sim + 0.9*pos
    np.fill_diagonal(raw, -1e9)                    # a word leans on its neighbours, never itself
    e=np.exp(raw - raw.max(axis=1, keepdims=True))
    return e/e.sum(axis=1, keepdims=True)
def heatmap(A, tokens, title):
    plt.figure(figsize=(1.1*len(tokens)+3, 1.0*len(tokens)+2))
    plt.imshow(A, cmap="YlOrRd", vmin=0, vmax=max(0.5, A.max()))
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right")
    plt.yticks(range(len(tokens)), tokens)
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            plt.text(j, i, format(A[i,j], ".2f"), ha="center", va="center",
                     fontsize=8, color="black" if A[i,j] < 0.45 else "white")
    plt.xlabel("...pays attention to this word"); plt.ylabel("This word...")
    plt.title(title); plt.colorbar(shrink=0.7); plt.tight_layout(); plt.show()

vocab, idx, V = space()
w = tricky_word.strip().lower()
print("The word under the microscope: '" + w + "'")
print("")
print("   Sentence 1: " + sentence_one)
print("   Sentence 2: " + sentence_two)
print("")
if w not in [t.lower() for t in words_of(sentence_one)] or w not in [t.lower() for t in words_of(sentence_two)]:
    problem("The word '" + w + "' needs to appear in BOTH sentences.")
    note("Edit the sentences above so they both contain it, then press play again.")
else:
    print("An old-style model stored ONE list of numbers for '" + w + "'.")
    print("So both sentences got the identical meaning - it could not tell them apart.")
    print("")
    c1 = [t for t in content(sentence_one) if t != w]
    c2 = [t for t in content(sentence_two) if t != w]
    print("But look at the company '" + w + "' keeps:")
    print("")
    print("   Sentence 1 neighbours: " + ", ".join(c1))
    print("   Sentence 2 neighbours: " + ", ".join(c2))
    print("")
    print("Those neighbours are completely different - and that is the clue")
    print("attention uses to tell the two meanings apart. Step 2 shows it happening.")

---
## Part 2: Watching attention work

The map below is the heart of it. Every row is a word asking *"who here matters to me?"*,
and the bright squares are its answer. Darker red = paying more attention.

In [ ]:
#@title ▶ Step 2 — Draw the attention map for my sentence { display-mode: "form" }
#@markdown Type a sentence. Short ones (5–10 words) are easiest to read.
my_sentence = "I sat on the bank of the river" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

# a small, meaningful word space so attention patterns are readable
CORPUS = [
 "went to the bank to deposit money in my account",
 "the bank gave me a loan and i paid money into the account",
 "money cash account loan deposit bank branch",
 "sat on the bank of the river watching the water",
 "the river bank was steep grassy and muddy",
 "river water fish boat grassy muddy bank",
 "the dog chased the cat around the garden",
 "my dog barked at the cat in the garden",
 "dog cat pet animal garden chased barked",
 "i love running football swimming and sport",
 "running sport exercise football healthy",
 "she ate rice and chicken for her lunch",
 "rice chicken food lunch tasty eating",
]
STOP = {"the","a","an","to","of","in","on","at","and","my","i","it","was","is","for","her","me"}
def content(text):
    keep = [w for w in words_of(text) if w not in STOP]
    return keep if len(keep) >= 4 else words_of(text)
def build_space(dim=16, window=4):
    T=[content(c) for c in CORPUS]
    vocab=sorted({w for t in T for w in t}); idx={w:i for i,w in enumerate(vocab)}
    nv=len(vocab); M=np.zeros((nv,nv))
    for t in T:
        for i,w in enumerate(t):
            for j in range(max(0,i-window), min(len(t),i+window+1)):
                if i!=j: M[idx[w],idx[t[j]]]+=1.0
    tot=M.sum(); rows=M.sum(1,keepdims=True); cols=M.sum(0,keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P=np.log((M*tot)/(rows*cols+1e-12)+1e-12)
    P[~np.isfinite(P)]=0; P[P<0]=0
    U,S,Vt=np.linalg.svd(P, full_matrices=False)
    V=U[:,:dim]*S[:dim]
    return vocab, idx, V/(np.linalg.norm(V,axis=1,keepdims=True)+1e-9)
def space():
    if "_SP3" not in globals(): globals()["_SP3"]=build_space()
    return globals()["_SP3"]
def attention(tokens):
    # how much each word looks at every other word: meaning similarity + nearness in the sentence
    vocab, idx, V = space()
    nT=len(tokens)
    vecs=np.array([V[idx[t]] if t in idx else np.zeros(V.shape[1]) for t in tokens])
    sim=vecs.dot(vecs.T)
    pos=np.array([[1.0/(1.0+abs(i-j)) for j in range(nT)] for i in range(nT)])
    raw=2.2*sim + 0.9*pos
    np.fill_diagonal(raw, -1e9)                    # a word leans on its neighbours, never itself
    e=np.exp(raw - raw.max(axis=1, keepdims=True))
    return e/e.sum(axis=1, keepdims=True)
def heatmap(A, tokens, title):
    plt.figure(figsize=(1.1*len(tokens)+3, 1.0*len(tokens)+2))
    plt.imshow(A, cmap="YlOrRd", vmin=0, vmax=max(0.5, A.max()))
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right")
    plt.yticks(range(len(tokens)), tokens)
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            plt.text(j, i, format(A[i,j], ".2f"), ha="center", va="center",
                     fontsize=8, color="black" if A[i,j] < 0.45 else "white")
    plt.xlabel("...pays attention to this word"); plt.ylabel("This word...")
    plt.title(title); plt.colorbar(shrink=0.7); plt.tight_layout(); plt.show()

toks = content(my_sentence)
if len(toks) < 2:
    problem("I need at least two real words. Try a longer sentence.")
else:
    toks = toks[:10]
    A = attention(toks)
    print("Reading: " + my_sentence.strip())
    print("")
    print("Words the model is tracking: " + ", ".join(toks))
    print("")
    heatmap(A, toks, "Who is looking at whom?")
    print("Row by row, here is who each word leaned on most:")
    print("")
    for i, t in enumerate(toks):
        j = int(np.argmax(A[i]))
        print("   " + t.ljust(12) + "-> looked mostly at " + toks[j].ljust(12) +
              "(" + format(A[i, j], ".0%") + " of its attention)")
    print("")
    print("So what? No rule told it which words go together.")
    print("It measured how often words keep company, and let each word")
    print("borrow meaning from whoever matters most.")

In [ ]:
#@title ▶ Step 3 — Compare the same word in two sentences { display-mode: "form" }
#@markdown Watch one word change meaning depending on its neighbours.
the_word = "bank" #@param {type:"string"}
sentence_one = "I went to the bank to deposit money" #@param {type:"string"}
sentence_two = "I sat on the bank of the river" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

# a small, meaningful word space so attention patterns are readable
CORPUS = [
 "went to the bank to deposit money in my account",
 "the bank gave me a loan and i paid money into the account",
 "money cash account loan deposit bank branch",
 "sat on the bank of the river watching the water",
 "the river bank was steep grassy and muddy",
 "river water fish boat grassy muddy bank",
 "the dog chased the cat around the garden",
 "my dog barked at the cat in the garden",
 "dog cat pet animal garden chased barked",
 "i love running football swimming and sport",
 "running sport exercise football healthy",
 "she ate rice and chicken for her lunch",
 "rice chicken food lunch tasty eating",
]
STOP = {"the","a","an","to","of","in","on","at","and","my","i","it","was","is","for","her","me"}
def content(text):
    keep = [w for w in words_of(text) if w not in STOP]
    return keep if len(keep) >= 4 else words_of(text)
def build_space(dim=16, window=4):
    T=[content(c) for c in CORPUS]
    vocab=sorted({w for t in T for w in t}); idx={w:i for i,w in enumerate(vocab)}
    nv=len(vocab); M=np.zeros((nv,nv))
    for t in T:
        for i,w in enumerate(t):
            for j in range(max(0,i-window), min(len(t),i+window+1)):
                if i!=j: M[idx[w],idx[t[j]]]+=1.0
    tot=M.sum(); rows=M.sum(1,keepdims=True); cols=M.sum(0,keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P=np.log((M*tot)/(rows*cols+1e-12)+1e-12)
    P[~np.isfinite(P)]=0; P[P<0]=0
    U,S,Vt=np.linalg.svd(P, full_matrices=False)
    V=U[:,:dim]*S[:dim]
    return vocab, idx, V/(np.linalg.norm(V,axis=1,keepdims=True)+1e-9)
def space():
    if "_SP3" not in globals(): globals()["_SP3"]=build_space()
    return globals()["_SP3"]
def attention(tokens):
    # how much each word looks at every other word: meaning similarity + nearness in the sentence
    vocab, idx, V = space()
    nT=len(tokens)
    vecs=np.array([V[idx[t]] if t in idx else np.zeros(V.shape[1]) for t in tokens])
    sim=vecs.dot(vecs.T)
    pos=np.array([[1.0/(1.0+abs(i-j)) for j in range(nT)] for i in range(nT)])
    raw=2.2*sim + 0.9*pos
    np.fill_diagonal(raw, -1e9)                    # a word leans on its neighbours, never itself
    e=np.exp(raw - raw.max(axis=1, keepdims=True))
    return e/e.sum(axis=1, keepdims=True)
def heatmap(A, tokens, title):
    plt.figure(figsize=(1.1*len(tokens)+3, 1.0*len(tokens)+2))
    plt.imshow(A, cmap="YlOrRd", vmin=0, vmax=max(0.5, A.max()))
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right")
    plt.yticks(range(len(tokens)), tokens)
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            plt.text(j, i, format(A[i,j], ".2f"), ha="center", va="center",
                     fontsize=8, color="black" if A[i,j] < 0.45 else "white")
    plt.xlabel("...pays attention to this word"); plt.ylabel("This word...")
    plt.title(title); plt.colorbar(shrink=0.7); plt.tight_layout(); plt.show()

vocab, idx, V = space()
w = the_word.strip().lower()
ok = True
for label, sent in [("Sentence 1", sentence_one), ("Sentence 2", sentence_two)]:
    if w not in content(sent):
        problem(label + " does not contain the word '" + w + "'."); ok = False
if ok:
    for label, sent in [("SENTENCE 1", sentence_one), ("SENTENCE 2", sentence_two)]:
        toks = content(sent)[:10]
        A = attention(toks)
        i = toks.index(w)
        order = np.argsort(-A[i])
        print(label + ": " + sent.strip())
        print("   '" + w + "' paid most attention to:")
        for j in order[:3]:
            if toks[j] != w:
                print("      " + toks[j].ljust(12) + format(A[i, j], ">6.0%"))
        print("")
    print("Same word. Totally different neighbours.")
    print("")
    print("So what? The model does not store two meanings of '" + w + "'.")
    print("It builds the right meaning on the spot, from whatever is nearby.")
    print("That is why it can read a word it has seen in a hundred different ways.")

---
## Part 3: Several readers at once

A real transformer does not read your sentence once. It reads it several times in
parallel, and each reading — called a **head** — pays attention to something different.

In [ ]:
#@title ▶ Step 4 — Read the same sentence three different ways { display-mode: "form" }
#@markdown Three "heads" read your sentence, each looking for something different.
my_sentence = "The dog chased the cat around the garden" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

# a small, meaningful word space so attention patterns are readable
CORPUS = [
 "went to the bank to deposit money in my account",
 "the bank gave me a loan and i paid money into the account",
 "money cash account loan deposit bank branch",
 "sat on the bank of the river watching the water",
 "the river bank was steep grassy and muddy",
 "river water fish boat grassy muddy bank",
 "the dog chased the cat around the garden",
 "my dog barked at the cat in the garden",
 "dog cat pet animal garden chased barked",
 "i love running football swimming and sport",
 "running sport exercise football healthy",
 "she ate rice and chicken for her lunch",
 "rice chicken food lunch tasty eating",
]
STOP = {"the","a","an","to","of","in","on","at","and","my","i","it","was","is","for","her","me"}
def content(text):
    keep = [w for w in words_of(text) if w not in STOP]
    return keep if len(keep) >= 4 else words_of(text)
def build_space(dim=16, window=4):
    T=[content(c) for c in CORPUS]
    vocab=sorted({w for t in T for w in t}); idx={w:i for i,w in enumerate(vocab)}
    nv=len(vocab); M=np.zeros((nv,nv))
    for t in T:
        for i,w in enumerate(t):
            for j in range(max(0,i-window), min(len(t),i+window+1)):
                if i!=j: M[idx[w],idx[t[j]]]+=1.0
    tot=M.sum(); rows=M.sum(1,keepdims=True); cols=M.sum(0,keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P=np.log((M*tot)/(rows*cols+1e-12)+1e-12)
    P[~np.isfinite(P)]=0; P[P<0]=0
    U,S,Vt=np.linalg.svd(P, full_matrices=False)
    V=U[:,:dim]*S[:dim]
    return vocab, idx, V/(np.linalg.norm(V,axis=1,keepdims=True)+1e-9)
def space():
    if "_SP3" not in globals(): globals()["_SP3"]=build_space()
    return globals()["_SP3"]
def attention(tokens):
    # how much each word looks at every other word: meaning similarity + nearness in the sentence
    vocab, idx, V = space()
    nT=len(tokens)
    vecs=np.array([V[idx[t]] if t in idx else np.zeros(V.shape[1]) for t in tokens])
    sim=vecs.dot(vecs.T)
    pos=np.array([[1.0/(1.0+abs(i-j)) for j in range(nT)] for i in range(nT)])
    raw=2.2*sim + 0.9*pos
    np.fill_diagonal(raw, -1e9)                    # a word leans on its neighbours, never itself
    e=np.exp(raw - raw.max(axis=1, keepdims=True))
    return e/e.sum(axis=1, keepdims=True)
def heatmap(A, tokens, title):
    plt.figure(figsize=(1.1*len(tokens)+3, 1.0*len(tokens)+2))
    plt.imshow(A, cmap="YlOrRd", vmin=0, vmax=max(0.5, A.max()))
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right")
    plt.yticks(range(len(tokens)), tokens)
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            plt.text(j, i, format(A[i,j], ".2f"), ha="center", va="center",
                     fontsize=8, color="black" if A[i,j] < 0.45 else "white")
    plt.xlabel("...pays attention to this word"); plt.ylabel("This word...")
    plt.title(title); plt.colorbar(shrink=0.7); plt.tight_layout(); plt.show()

toks = content(my_sentence)[:9]
if len(toks) < 3:
    problem("I need at least three real words. Try a longer sentence.")
else:
    vocab, idx, V = space()
    nT = len(toks)
    vecs = np.array([V[idx[t]] if t in idx else np.zeros(V.shape[1]) for t in toks])
    sim = vecs.dot(vecs.T)
    pos = np.array([[1.0 / (1.0 + abs(i - j)) for j in range(nT)] for i in range(nT)])
    nxt = np.array([[1.0 if j == i + 1 else 0.0 for j in range(nT)] for i in range(nT)])
    def soft(raw):
        np.fill_diagonal(raw, -1e9)
        e = np.exp(raw - raw.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)
    heads = [("Head 1 - the meaning reader", soft(3.0 * sim), "looks for words about the same topic"),
             ("Head 2 - the neighbour reader", soft(2.5 * pos), "looks at whatever sits closest"),
             ("Head 3 - the what-comes-next reader", soft(3.0 * nxt + 0.4 * pos), "looks at the word that follows")]
    print("Reading: " + my_sentence.strip())
    print("")
    for name, A, why in heads:
        print(name + " - " + why)
        for i, t in enumerate(toks):
            j = int(np.argmax(A[i]))
            print("   " + t.ljust(12) + "-> " + toks[j])
        print("")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    for ax, (name, A, why) in zip(axes, heads):
        ax.imshow(A, cmap="YlOrRd", vmin=0, vmax=max(0.5, A.max()))
        ax.set_xticks(range(nT)); ax.set_xticklabels(toks, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(nT)); ax.set_yticklabels(toks, fontsize=8)
        ax.set_title(name.split(" - ")[0] + "\n" + name.split(" - ")[1], fontsize=9)
    plt.tight_layout(); plt.show()
    print("So what? One reading is never enough. Grammar, topic and word order")
    print("are different questions, so the model asks all of them at once")
    print("and adds the answers together.")

---
## Summary & Key Takeaways

1. **The problem** — a word's meaning depends on its neighbours, so one fixed meaning per word fails.
2. **Attention** — every word asks "who here matters to me?" and borrows meaning from the answer.
3. **The map** — bright squares show where a word is looking.
4. **Many heads** — the model reads the same sentence several ways at once: topic, position, what follows.

> This is the mechanism behind the "T" in GPT — the **Transformer**.

### Next up
How to give instructions a model will actually follow — **prompt engineering**.